# CrewAI — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-2/crewai.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicAI-labs/learn-notebooks/blob/main/ai-agents/6-2/crewai.ipynb)

Companion to: [CrewAI](https://anacodicai.org/learn/ai-agents/6-2/crewai)


## What you'll build

The **real `crewai` library** — not a simulation. Same trip-planning example
as the article's real-world comparison: a **Budget Analyst**, a **Flights
Agent**, and a **Trip Coordinator** that can delegate between them. We
construct real `Agent`, `Task`, and `Crew` objects and inspect exactly what
CrewAI built: roles, goals, the default LLM config each agent picked up, and
the sequential process graph connecting tasks together.

**No API key needed for any of this** — creating agents, tasks, and a crew
never calls a model. Only the very last, clearly-marked bonus cell
(`crew.kickoff()`) needs a real key, because that's the one line that
actually asks an LLM to do the work. `pip install crewai` if running locally.

### Setup

Two environment quirks CrewAI runs into outside a plain laptop/Colab setup —
both copy-pasteable fixes:

1. CrewAI uses `chromadb` for memory, which needs a newer `sqlite3` than some
   systems ship with. `RuntimeError: Your system has an unsupported version
   of sqlite3` is fixed by swapping in `pysqlite3-binary`'s bundled sqlite3
   before crewai (transitively) imports the real one.
2. CrewAI stores task-output history in a local SQLite file under your home
   directory. If that home directory lives on a network filesystem (common
   on shared servers and HPC clusters), SQLite's WAL locking fails with
   `Database initialization error: locking protocol`. Pointing
   `XDG_DATA_HOME` at a local, non-network directory fixes it — harmless on
   a normal machine or Colab, where the default is already local.

In [1]:
# %pip install -q crewai pysqlite3-binary
import os
import sys
import tempfile

os.environ.setdefault("XDG_DATA_HOME", tempfile.gettempdir())

try:
    import pysqlite3
    sys.modules["sqlite3"] = pysqlite3
except ImportError:
    pass  # only needed on systems with sqlite3 < 3.35

import crewai
from crewai import Agent, Task, Crew, Process

print("real crewai", crewai.__version__, "imported")

real crewai 1.15.14 imported


## Step 1 — a real Agent

`role`, `goal`, and `backstory` aren't just labels for a printout — CrewAI
folds them into the system prompt it builds for whichever LLM ends up
running this agent. Notice `allow_delegation=False`: this agent can't hand
work off to teammates, which matters once a crew has more than one.

In [2]:
budget_analyst = Agent(
    role="Budget Analyst",
    goal="Keep the total trip cost under the group's budget cap",
    backstory=(
        "Tracks every planned expense against the cap and flags anything "
        "that would blow it, with a one-line reason."
    ),
    allow_delegation=False,
    verbose=False,
)

print("role:     ", budget_analyst.role)
print("goal:     ", budget_analyst.goal)
print("delegates:", budget_analyst.allow_delegation)

role:      Budget Analyst
goal:      Keep the total trip cost under the group's budget cap
delegates: False


## Step 2 — every agent already has a real, inspectable LLM config

We never set an LLM. CrewAI picked a real default (`OpenAICompletion`,
`gpt-4.1-mini`) and attached it to the agent — with `api_key=None`, because
nothing has asked it to actually call out yet. Constructing an agent and
*running* an agent are two separate steps in CrewAI, same as LangGraph
separates compiling a graph from invoking it.

In [3]:
print(type(budget_analyst.llm).__name__)
print("model:  ", budget_analyst.llm.model)
print("api_key:", budget_analyst.llm.api_key)

assert budget_analyst.llm.api_key is None
print("\nassert passed: the agent has a real LLM object attached, unauthenticated")

OpenAICompletion
model:   gpt-4.1-mini
api_key: None

assert passed: the agent has a real LLM object attached, unauthenticated


## Step 3 — a real Task, bound to that agent

`{trip}` and `{budget_cap}` are template slots, same idea as LangChain's
`{topic}` — CrewAI fills them in from the inputs passed to `kickoff()`.

In [4]:
budget_task = Task(
    description=(
        "Given a proposed itinerary for {trip}, confirm it stays under the "
        "{budget_cap} per-person budget cap."
    ),
    expected_output="APPROVE, or OVER-BUDGET plus the amount over.",
    agent=budget_analyst,
)

print("assigned agent:", budget_task.agent.role)
print("expected output:", budget_task.expected_output)

assigned agent: Budget Analyst
expected output: APPROVE, or OVER-BUDGET plus the amount over.


## Step 4 — a Crew: agents plus tasks plus a process

`Process.sequential` means CrewAI runs tasks in the order given, each one
seeing the previous task's output. With only one task here it's trivial, but
this is the exact same object a full trip-planning crew — budget, flights,
hotels, activities — would use for a fixed pipeline.

In [5]:
crew = Crew(
    agents=[budget_analyst],
    tasks=[budget_task],
    process=Process.sequential,
    verbose=False,
)

print("agents in crew:", [a.role for a in crew.agents])
print("tasks in crew: ", len(crew.tasks))
print("process:       ", crew.process)

assert crew.process == Process.sequential
assert len(crew.agents) == 1 and len(crew.tasks) == 1
print("\nasserts passed: a real Crew object was constructed from a real Agent and Task")

agents in crew: ['Budget Analyst']
tasks in crew:  1
process:        Process.sequential

asserts passed: a real Crew object was constructed from a real Agent and Task


## Step 5 — a second agent, and delegation

Add a **Flights Agent** for searching options, plus a **Trip Coordinator**
who CAN delegate — the shape CrewAI actually exists for: specialized agents
cooperating on a pipeline, not one agent alone.

In [6]:
flights_agent = Agent(
    role="Flights Agent",
    goal="Find flight options that fit the group's dates and the budget cap",
    backstory="Searches routes and fares, and flags tradeoffs between price and convenience.",
    allow_delegation=False,
    verbose=False,
)

flights_task = Task(
    description="Find flight options for {trip} that fit under {budget_cap} per person.",
    expected_output="A short list of options with price and total travel time.",
    agent=flights_agent,
)

coordinator = Agent(
    role="Trip Coordinator",
    goal="Assemble a final itinerary by pulling in the right specialist for each piece",
    backstory="Decides which specialist a request actually needs and merges their answers.",
    allow_delegation=True,
    verbose=False,
)

two_agent_crew = Crew(
    agents=[budget_analyst, flights_agent],
    tasks=[budget_task, flights_task],
    process=Process.sequential,
    verbose=False,
)

print("agents:", [a.role for a in two_agent_crew.agents])
print("coordinator can delegate:", coordinator.allow_delegation)
assert len(two_agent_crew.agents) == 2
print("\nassert passed: a two-agent sequential crew was constructed")

agents: ['Budget Analyst', 'Flights Agent']
coordinator can delegate: True

assert passed: a two-agent sequential crew was constructed


## Bonus (needs a real API key) — actually run it

Everything above builds and inspects the real object model with zero network
calls. This cell is the one line that needs a key — set `OPENAI_API_KEY` (or
configure `budget_analyst.llm` for a different provider) and uncomment it to
see the crew actually produce a real answer.

In [7]:
import os

if os.environ.get("OPENAI_API_KEY"):
    result = crew.kickoff(inputs={"trip": "a 5-day Barcelona trip for 4 people", "budget_cap": "$900"})
    print(result)
else:
    print("Set OPENAI_API_KEY and re-run this cell to see crew.kickoff() "
          "actually call the LLM. Skipped here since this notebook promises "
          "to run with no key required.")

Set OPENAI_API_KEY and re-run this cell to see crew.kickoff() actually call the LLM. Skipped here since this notebook promises to run with no key required.


## Try it yourself

- Add a **Hotels Agent** and an **Activities Planner** to `two_agent_crew`,
  matching the full four-specialist trip crew described in the article.
- Swap `Process.sequential` for `Process.hierarchical` with the coordinator
  as `manager_llm` — the article's own contrast: sequential is right when the
  pipeline is fixed (search flights → budget-filter → hotel-match → plan
  activities); hierarchical is right when a manager should dynamically skip
  specialists a given query doesn't need (an activities-only query may skip
  the budget agent entirely).
- Set `budget_analyst.llm.model = "gpt-4o"` (or point it at a different
  provider entirely) before `kickoff()` — notice the Agent/Task/Crew code
  above doesn't change at all, same as swapping the model in the LangChain
  notebook.